# GPT-2 Architecture: From Building Blocks to Full Model

This notebook constructs the complete GPT-2 architecture from scratch, piece by piece.

We will:

1. **Re-implement every building block inline** -- multi-head causal self-attention, the pre-norm transformer block, and the feed-forward network.
2. **Stack transformer blocks** into a full GPT-2 model with token embeddings, positional embeddings, a final LayerNorm, and a linear output head with weight tying.
3. **Run a complete forward pass** with a tiny model (`vocab_size=256, embed_dim=64, n_heads=4, n_layers=2, max_seq_len=128`) and inspect the output.
4. **Count parameters** analytically and compare against all four official GPT-2 variants (Small through XL).
5. **Visualize** parameter distributions, activation norms, and the effects of depth and width through ablation experiments.

Every cell is self-contained -- no external `.py` files are imported. Only `torch`, `numpy`, and `matplotlib` are used.

In [ ]:
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Default tiny dimensions
VOCAB_SIZE = 256
EMBED_DIM = 64
N_HEADS = 4
N_LAYERS = 2
MAX_SEQ_LEN = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Tiny config: vocab={VOCAB_SIZE}, dim={EMBED_DIM}, heads={N_HEADS}, layers={N_LAYERS}, max_seq={MAX_SEQ_LEN}")

---
## Part 1: Building Blocks

### 1.1 Multi-Head Causal Self-Attention

GPT-2 uses **causal (masked) self-attention**: each token can only attend to itself and all preceding tokens. This is what makes it *autoregressive* -- it can generate text left-to-right without peeking at the future.

The attention computation for each head is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

where $M$ is the causal mask (upper-triangular entries set to $-\infty$).

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head causal self-attention, as used in GPT-2."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads

        # Combined QKV projection (GPT-2 style)
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        # Output projection
        self.c_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape  # batch, sequence length, embedding dim

        # Project to Q, K, V
        qkv = self.c_attn(x)  # (B, T, 3*C)
        q, k, v = qkv.split(self.embed_dim, dim=2)  # each (B, T, C)

        # Reshape into (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention with causal mask
        scale = 1.0 / (self.head_dim ** 0.5)
        attn = (q @ k.transpose(-2, -1)) * scale  # (B, H, T, T)

        # Causal mask: prevent attending to future positions
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn = F.softmax(attn, dim=-1)  # (B, H, T, T)

        # Weighted sum of values
        out = attn @ v  # (B, H, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)

        # Output projection
        out = self.c_proj(out)
        return out


# Quick test
mha = MultiHeadAttention(EMBED_DIM, N_HEADS)
test_input = torch.randn(2, 10, EMBED_DIM)
test_output = mha(test_input)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Parameters:   {sum(p.numel() for p in mha.parameters()):,}")

The multi-head attention module preserves the shape of its input: `(B, T, C)` in, `(B, T, C)` out. Internally it splits the embedding dimension across heads, computes attention independently per head, then concatenates and projects back.

### 1.2 Feed-Forward Network (MLP)

Each transformer block contains a position-wise feed-forward network that expands the representation to 4x the embedding dimension, applies GELU activation, then projects back:

$$\text{FFN}(x) = \text{GELU}(xW_1 + b_1)W_2 + b_2$$

GPT-2 uses the GELU activation instead of ReLU -- it provides a smooth, non-monotonic nonlinearity.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network with GELU activation."""

    def __init__(self, embed_dim):
        super().__init__()
        self.c_fc = nn.Linear(embed_dim, 4 * embed_dim)
        self.c_proj = nn.Linear(4 * embed_dim, embed_dim)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        return x


ff = FeedForward(EMBED_DIM)
test_out = ff(test_input)
print(f"FFN input shape:  {test_input.shape}")
print(f"FFN output shape: {test_out.shape}")
print(f"FFN parameters:   {sum(p.numel() for p in ff.parameters()):,}")
print(f"  - c_fc:   {EMBED_DIM} -> {4*EMBED_DIM}  = {EMBED_DIM * 4 * EMBED_DIM + 4 * EMBED_DIM:,} params")
print(f"  - c_proj: {4*EMBED_DIM} -> {EMBED_DIM}  = {4 * EMBED_DIM * EMBED_DIM + EMBED_DIM:,} params")

### 1.3 Transformer Block (Pre-Norm)

GPT-2 uses **pre-norm** residual connections (LayerNorm *before* each sub-layer), unlike the original Transformer which uses post-norm. The pattern is:

```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

Pre-norm makes training more stable, especially for deep networks, because the residual path stays clean.

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm transformer block as used in GPT-2."""

    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln_1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads)
        self.ln_2 = nn.LayerNorm(embed_dim)
        self.mlp = FeedForward(embed_dim)

    def forward(self, x):
        # Pre-norm: LayerNorm -> Attention -> Residual
        x = x + self.attn(self.ln_1(x))
        # Pre-norm: LayerNorm -> FFN -> Residual
        x = x + self.mlp(self.ln_2(x))
        return x


block = TransformerBlock(EMBED_DIM, N_HEADS)
block_out = block(test_input)
print(f"Block input shape:  {test_input.shape}")
print(f"Block output shape: {block_out.shape}")
print(f"Block parameters:   {sum(p.numel() for p in block.parameters()):,}")

Each transformer block preserves the tensor shape. The residual connections mean the output is always `input + transformation(input)`, which gives gradients a direct path backward through the network.

---
## Part 2: Stacking Transformer Blocks

GPT-2 stacks $N$ identical transformer blocks using `nn.ModuleList`. Each block has the **same architecture** but learns **different parameters**. The depth of the model -- how many blocks are stacked -- is one of the primary scaling knobs.

In [ ]:
# Demonstrate stacking blocks
n_layers_demo = 4
blocks = nn.ModuleList([TransformerBlock(EMBED_DIM, N_HEADS) for _ in range(n_layers_demo)])

print(f"Number of blocks: {len(blocks)}")
print(f"Total parameters across all blocks: {sum(p.numel() for p in blocks.parameters()):,}")
print()

# Show that each block has independent parameters
for i, block in enumerate(blocks):
    n_params = sum(p.numel() for p in block.parameters())
    w_norm = block.attn.c_attn.weight.norm().item()
    print(f"  Block {i}: {n_params:,} params, attn weight norm = {w_norm:.4f}")

# Run input through the stack sequentially
print()
x = torch.randn(1, 16, EMBED_DIM)  # (batch=1, seq_len=16, embed_dim)
print(f"Input shape: {x.shape}")

activations = [x.detach().clone()]
for i, block in enumerate(blocks):
    x = block(x)
    activations.append(x.detach().clone())
    print(f"After block {i}: shape = {x.shape}, norm = {x.norm():.4f}")

print(f"\nOutput shape: {x.shape}  (same as input -- shape is preserved throughout)")

---
## Part 3: Final LayerNorm + Linear Projection

After all transformer blocks, GPT-2 applies:

1. **A final LayerNorm** (`ln_f`) to stabilize the output representation.
2. **A linear projection** that maps from `embed_dim` to `vocab_size`, producing logits for each token in the vocabulary.

### Weight Tying

A key design choice: the output projection matrix shares its weights with the token embedding matrix. This is called **weight tying**. Instead of learning a separate `(embed_dim, vocab_size)` matrix for the output head, we reuse the token embedding weights `(vocab_size, embed_dim)`.

Benefits:
- Reduces parameter count significantly (by `vocab_size * embed_dim` parameters)
- Creates symmetry: tokens that are embedded similarly will also have similar output logits
- Acts as a regularizer that improves generalization

In [ ]:
# Demonstrate weight tying
token_emb = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
head = nn.Linear(EMBED_DIM, VOCAB_SIZE, bias=False)

print("Before weight tying:")
print(f"  token_emb.weight shape: {token_emb.weight.shape}")
print(f"  head.weight shape:      {head.weight.shape}")
print(f"  Same object? {token_emb.weight is head.weight}")

# Tie weights
head.weight = token_emb.weight

print("\nAfter weight tying:")
print(f"  Same object? {token_emb.weight is head.weight}")
print(f"  Parameter savings: {VOCAB_SIZE * EMBED_DIM:,} parameters")

---
## Part 4: The Full GPT-2 Model

Now we assemble everything into a single `GPT2` class. The architecture is:

```
Input token IDs
    -> Token Embedding + Positional Embedding
    -> N x TransformerBlock (pre-norm: LN -> Attn -> Residual -> LN -> FFN -> Residual)
    -> Final LayerNorm
    -> Linear projection to vocab (weight-tied with token embedding)
    -> Output logits
```

In [ ]:
class GPT2(nn.Module):
    """Full GPT-2 model."""

    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, max_seq_len):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.max_seq_len = max_seq_len

        # Token and positional embeddings
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embed_dim)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, n_heads) for _ in range(n_layers)]
        )

        # Final layer norm
        self.ln_f = nn.LayerNorm(embed_dim)

        # Output projection (no bias)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)

        # Weight tying: output projection shares weights with token embedding
        self.head.weight = self.token_emb.weight

    def forward(self, idx):
        """
        Args:
            idx: (B, T) tensor of token indices
        Returns:
            logits: (B, T, vocab_size) tensor of next-token logits
        """
        B, T = idx.shape
        assert T <= self.max_seq_len, f"Sequence length {T} exceeds max {self.max_seq_len}"

        # Embeddings
        tok_emb = self.token_emb(idx)  # (B, T, C)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb  # (B, T, C) -- broadcasting adds pos_emb to each batch

        # Transformer blocks
        for block in self.blocks:
            x = block(x)

        # Final layer norm + projection
        x = self.ln_f(x)  # (B, T, C)
        logits = self.head(x)  # (B, T, vocab_size)

        return logits


# Instantiate the tiny model
torch.manual_seed(42)
model = GPT2(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    max_seq_len=MAX_SEQ_LEN,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Tiny GPT-2 model created")
print(f"  vocab_size:   {VOCAB_SIZE}")
print(f"  embed_dim:    {EMBED_DIM}")
print(f"  n_heads:      {N_HEADS}")
print(f"  n_layers:     {N_LAYERS}")
print(f"  max_seq_len:  {MAX_SEQ_LEN}")
print(f"  Total params: {total_params:,}")
print()
print(model)

---
## Part 5: GPT-2 Configuration Variants

OpenAI released four sizes of GPT-2. They all share the same architecture -- the only things that change are the depth (number of layers), width (embedding dimension), and number of attention heads.

In [ ]:
from dataclasses import dataclass


@dataclass
class GPT2Config:
    name: str
    n_layers: int
    embed_dim: int
    n_heads: int
    vocab_size: int = 50257  # GPT-2 BPE vocabulary
    max_seq_len: int = 1024  # GPT-2 context window


configs = [
    GPT2Config(name="GPT-2 Small",  n_layers=12, embed_dim=768,  n_heads=12),
    GPT2Config(name="GPT-2 Medium", n_layers=24, embed_dim=1024, n_heads=16),
    GPT2Config(name="GPT-2 Large",  n_layers=36, embed_dim=1280, n_heads=20),
    GPT2Config(name="GPT-2 XL",     n_layers=48, embed_dim=1600, n_heads=25),
]


def count_params(cfg):
    """Count parameters for a GPT-2 config analytically."""
    V, D, H, L, S = cfg.vocab_size, cfg.embed_dim, cfg.n_heads, cfg.n_layers, cfg.max_seq_len

    # Token embedding: V * D (shared with output head due to weight tying)
    token_emb = V * D
    # Positional embedding: S * D
    pos_emb = S * D

    # Per transformer block:
    #   LN_1: 2*D (weight + bias)
    #   Attention c_attn (QKV): D * 3D + 3D = 3D^2 + 3D
    #   Attention c_proj: D * D + D = D^2 + D
    #   LN_2: 2*D
    #   FFN c_fc: D * 4D + 4D = 4D^2 + 4D
    #   FFN c_proj: 4D * D + D = 4D^2 + D
    per_block = (2 * D) + (3 * D * D + 3 * D) + (D * D + D) + (2 * D) + (4 * D * D + 4 * D) + (4 * D * D + D)
    all_blocks = L * per_block

    # Final LayerNorm: 2*D
    final_ln = 2 * D

    # Output head: weight-tied with token_emb, so 0 additional params
    output_head = 0

    total = token_emb + pos_emb + all_blocks + final_ln + output_head
    return {
        "token_emb": token_emb,
        "pos_emb": pos_emb,
        "blocks": all_blocks,
        "final_ln": final_ln,
        "output_head": output_head,
        "total": total,
    }


print(f"{'Model':<16} {'Layers':>6} {'Dim':>6} {'Heads':>6} {'Parameters':>14}")
print("-" * 56)
for cfg in configs:
    params = count_params(cfg)
    total = params["total"]
    if total >= 1e9:
        label = f"{total/1e9:.2f}B"
    else:
        label = f"{total/1e6:.0f}M"
    print(f"{cfg.name:<16} {cfg.n_layers:>6} {cfg.embed_dim:>6} {cfg.n_heads:>6} {total:>12,}  ({label})")

Notice how parameter count grows roughly quadratically with embedding dimension (because of the `D^2` terms in attention and FFN) and linearly with the number of layers.

---
## Part 6: Full Forward Pass

Let us run a complete forward pass through our tiny model and inspect every stage of the output.

In [ ]:
# Create random token input and run forward pass
torch.manual_seed(123)
batch_size = 2
seq_len = 32
input_ids = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len), device=device)

print(f"Input token IDs shape: {input_ids.shape}")
print(f"First sequence (first 10 tokens): {input_ids[0, :10].tolist()}")
print()

# Forward pass
model.eval()
with torch.no_grad():
    logits = model(input_ids)

print(f"Output logits shape: {logits.shape}")
print(f"  - Batch size:      {logits.shape[0]}")
print(f"  - Sequence length: {logits.shape[1]}")
print(f"  - Vocabulary size: {logits.shape[2]}")
print()
print(f"Logits range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")
print(f"Logits mean:  {logits.mean().item():.4f}")
print(f"Logits std:   {logits.std().item():.4f}")

In [ ]:
# Apply softmax to get next-token probabilities
probs = F.softmax(logits, dim=-1)  # (B, T, vocab_size)

print(f"Probability distribution shape: {probs.shape}")
print(f"Sum of probs for position 0, batch 0: {probs[0, 0].sum().item():.6f} (should be 1.0)")
print()

# Look at the probability distribution for the last token (next-token prediction)
last_probs = probs[0, -1]  # probabilities for predicting the token after position 31
top_k = 10
top_probs, top_indices = torch.topk(last_probs, top_k)

print(f"Top {top_k} predicted next tokens (batch 0, after position {seq_len-1}):")
for i in range(top_k):
    print(f"  Token {top_indices[i].item():>4}: prob = {top_probs[i].item():.6f}")

print(f"\nNote: With random weights, the distribution is nearly uniform.")
print(f"Uniform probability would be: {1.0 / VOCAB_SIZE:.6f}")
print(f"Max predicted probability:    {top_probs[0].item():.6f}")

With random (untrained) weights, the model produces a nearly uniform distribution over the vocabulary. Training adjusts these weights so that the model assigns high probability to the correct next token.

---
## Part 7: Parameter Counting Formula

Let us derive exactly where every parameter lives in GPT-2, then verify our formula against the actual PyTorch model.

In [ ]:
def detailed_param_breakdown(vocab_size, embed_dim, n_heads, n_layers, max_seq_len):
    """Break down parameter count by component."""
    D = embed_dim
    V = vocab_size
    L = n_layers
    S = max_seq_len

    components = {}

    # Embeddings
    components["Token embedding (V*D)"] = V * D
    components["Positional embedding (S*D)"] = S * D

    # Per-block components (multiply by L)
    components["Attention QKV weights (L * 3*D*D)"] = L * 3 * D * D
    components["Attention QKV biases (L * 3*D)"] = L * 3 * D
    components["Attention output weights (L * D*D)"] = L * D * D
    components["Attention output biases (L * D)"] = L * D
    components["Attention LayerNorm (L * 2*D)"] = L * 2 * D
    components["FFN expand weights (L * D*4D)"] = L * D * 4 * D
    components["FFN expand biases (L * 4D)"] = L * 4 * D
    components["FFN contract weights (L * 4D*D)"] = L * 4 * D * D
    components["FFN contract biases (L * D)"] = L * D
    components["FFN LayerNorm (L * 2*D)"] = L * 2 * D

    # Final
    components["Final LayerNorm (2*D)"] = 2 * D
    components["Output head (weight-tied)"] = 0

    total = sum(components.values())
    return components, total


# Breakdown for our tiny model
components, formula_total = detailed_param_breakdown(VOCAB_SIZE, EMBED_DIM, N_HEADS, N_LAYERS, MAX_SEQ_LEN)
actual_total = sum(p.numel() for p in model.parameters())

print("Parameter breakdown (tiny model):")
print("=" * 60)
for name, count in components.items():
    pct = 100.0 * count / formula_total if formula_total > 0 else 0
    print(f"  {name:<45} {count:>8,}  ({pct:>5.1f}%)")
print("=" * 60)
print(f"  {'Formula total':<45} {formula_total:>8,}")
print(f"  {'Actual PyTorch total':<45} {actual_total:>8,}")
print(f"  {'Match?':<45} {formula_total == actual_total}")

In [ ]:
# Approximate formula: total ~ 12 * L * D^2
# This comes from: per block ~ 12*D^2 (4*D^2 attn + 8*D^2 FFN, ignoring biases and LN)

print("Approximate formula: total ~ 12 * n_layers * embed_dim^2")
print("(This captures the dominant D^2 terms from attention and FFN.)")
print()
print(f"{'Model':<16} {'Exact':>14} {'Approx (12LD^2)':>16} {'Error':>8}")
print("-" * 60)

for cfg in configs:
    exact = count_params(cfg)["total"]
    approx = 12 * cfg.n_layers * cfg.embed_dim ** 2
    error = 100.0 * abs(exact - approx) / exact
    print(f"{cfg.name:<16} {exact:>14,} {approx:>16,} {error:>7.1f}%")

print()
print("The approximation is within ~25-30% because it ignores embedding params,")
print("biases, and LayerNorm -- which matter more for smaller models.")

In [ ]:
# Full breakdown for all GPT-2 variants
print("Detailed breakdown for each GPT-2 variant:")
print()

for cfg in configs:
    params = count_params(cfg)
    total = params["total"]
    print(f"{cfg.name} ({cfg.n_layers}L, {cfg.embed_dim}D, {cfg.n_heads}H):")
    print(f"  Token embeddings:   {params['token_emb']:>12,}  ({100*params['token_emb']/total:.1f}%)")
    print(f"  Position embeddings:{params['pos_emb']:>12,}  ({100*params['pos_emb']/total:.1f}%)")
    print(f"  Transformer blocks: {params['blocks']:>12,}  ({100*params['blocks']/total:.1f}%)")
    print(f"  Final LayerNorm:    {params['final_ln']:>12,}  ({100*params['final_ln']/total:.1f}%)")
    print(f"  Total:              {total:>12,}")
    print()

---
## Part 8: Visualizations

### 8.1 Parameter Distribution Pie Chart

Where do all the parameters live? Let us break them into four categories: embeddings, attention, FFN, and other (LayerNorm).

In [ ]:
def categorize_params(cfg):
    """Categorize parameters into embeddings, attention, FFN, and other."""
    D = cfg.embed_dim
    V = cfg.vocab_size
    L = cfg.n_layers
    S = cfg.max_seq_len

    embeddings = V * D + S * D
    attention = L * (3 * D * D + 3 * D + D * D + D)  # QKV + output proj
    ffn = L * (D * 4 * D + 4 * D + 4 * D * D + D)   # expand + contract
    layernorm = L * (2 * D + 2 * D) + 2 * D          # per-block LNs + final LN

    return {
        "Embeddings": embeddings,
        "Attention": attention,
        "FFN": ffn,
        "LayerNorm": layernorm,
    }


fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

for ax, cfg in zip(axes, configs):
    cats = categorize_params(cfg)
    labels = list(cats.keys())
    sizes = list(cats.values())
    total = sum(sizes)

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        autopct="%1.1f%%",
        colors=colors,
        textprops={"fontsize": 8},
        pctdistance=0.75,
    )
    for t in autotexts:
        t.set_fontsize(7)
    if total >= 1e9:
        ax.set_title(f"{cfg.name}\n({total/1e9:.2f}B params)", fontsize=10)
    else:
        ax.set_title(f"{cfg.name}\n({total/1e6:.0f}M params)", fontsize=10)

fig.suptitle("Parameter Distribution Across GPT-2 Variants", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Observation: As models grow, FFN and attention dominate. Embedding share shrinks.")

### 8.2 Layer-by-Layer Activation Norms

Let us trace the L2 norm of the hidden representation after each transformer block. This shows how the signal magnitude evolves through the network.

In [ ]:
# Collect activation norms layer by layer
model.eval()
torch.manual_seed(42)
probe_input = torch.randint(0, VOCAB_SIZE, (1, 32), device=device)

activation_norms = []
layer_names = []

with torch.no_grad():
    # Embedding
    tok_emb = model.token_emb(probe_input)
    pos_emb = model.pos_emb(torch.arange(probe_input.shape[1], device=device))
    x = tok_emb + pos_emb
    activation_norms.append(x.norm().item())
    layer_names.append("Embedding")

    # Each transformer block
    for i, block in enumerate(model.blocks):
        x = block(x)
        activation_norms.append(x.norm().item())
        layer_names.append(f"Block {i}")

    # Final LayerNorm
    x = model.ln_f(x)
    activation_norms.append(x.norm().item())
    layer_names.append("Final LN")

    # Output logits
    logits_probe = model.head(x)
    activation_norms.append(logits_probe.norm().item())
    layer_names.append("Logits")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(activation_norms)), activation_norms, color="#4C72B0", edgecolor="white")
ax.set_xticks(range(len(layer_names)))
ax.set_xticklabels(layer_names, rotation=30, ha="right")
ax.set_ylabel("L2 Norm of Activations")
ax.set_title("Activation Norms Through the GPT-2 Pipeline", fontweight="bold")
ax.grid(axis="y", alpha=0.3)

# Annotate values
for bar, val in zip(bars, activation_norms):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

print("The residual connections cause activation norms to grow gradually through the network.")
print("The final LayerNorm re-normalizes before the output projection.")

---
## Part 9: Ablation Studies

### 9.1 Effect of Depth: 1 vs 2 vs 4 Layers

How does the number of transformer blocks affect the model's output? With untrained weights, we can measure the **entropy** of the output distribution -- deeper models tend to produce more structured (lower-entropy) distributions even before training.

In [ ]:
def compute_output_entropy(mdl, inp_ids):
    """Compute average entropy of the output distribution."""
    mdl.eval()
    with torch.no_grad():
        out_logits = mdl(inp_ids)
        out_probs = F.softmax(out_logits, dim=-1)
        log_p = torch.log(out_probs + 1e-10)
        entropy = -(out_probs * log_p).sum(dim=-1)  # (B, T)
        return entropy.mean().item()


depth_values = [1, 2, 4]
entropies = []
param_counts = []

torch.manual_seed(42)
test_ids = torch.randint(0, VOCAB_SIZE, (4, 32), device=device)

for n_l in depth_values:
    torch.manual_seed(42)
    m = GPT2(VOCAB_SIZE, EMBED_DIM, N_HEADS, n_l, MAX_SEQ_LEN).to(device)
    ent = compute_output_entropy(m, test_ids)
    n_p = sum(p.numel() for p in m.parameters())
    entropies.append(ent)
    param_counts.append(n_p)
    print(f"Depth={n_l}: entropy={ent:.4f}, params={n_p:,}")

max_entropy = np.log(VOCAB_SIZE)
print(f"\nMaximum possible entropy (uniform over {VOCAB_SIZE} tokens): {max_entropy:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Entropy vs depth
ax1.bar([str(d) for d in depth_values], entropies, color="#DD8452", edgecolor="white")
ax1.axhline(y=max_entropy, color="gray", linestyle="--", alpha=0.7, label=f"Max entropy ({max_entropy:.2f})")
ax1.set_xlabel("Number of Layers")
ax1.set_ylabel("Mean Output Entropy")
ax1.set_title("Output Entropy vs Depth", fontweight="bold")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Parameter count vs depth
ax2.bar([str(d) for d in depth_values], [p / 1000 for p in param_counts], color="#55A868", edgecolor="white")
ax2.set_xlabel("Number of Layers")
ax2.set_ylabel("Parameters (thousands)")
ax2.set_title("Parameter Count vs Depth", fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

# Annotate bars
for i, (e, p) in enumerate(zip(entropies, param_counts)):
    ax1.text(i, e + 0.02, f"{e:.3f}", ha="center", va="bottom", fontsize=9)
    ax2.text(i, p / 1000 + 0.5, f"{p:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

### 9.2 Effect of Width: embed_dim 32 vs 64 vs 128

Width (embedding dimension) affects both the representational capacity and the parameter count. Because parameters scale as $D^2$, doubling the width roughly quadruples the parameter count per block.

In [ ]:
width_values = [32, 64, 128]
width_entropies = []
width_param_counts = []

for dim in width_values:
    n_h = max(1, dim // 16)  # keep head_dim ~ 16
    torch.manual_seed(42)
    m = GPT2(VOCAB_SIZE, dim, n_h, N_LAYERS, MAX_SEQ_LEN).to(device)
    ent = compute_output_entropy(m, test_ids)
    n_p = sum(p.numel() for p in m.parameters())
    width_entropies.append(ent)
    width_param_counts.append(n_p)
    print(f"Width={dim}, heads={n_h}: entropy={ent:.4f}, params={n_p:,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Entropy vs width
ax1.bar([str(w) for w in width_values], width_entropies, color="#C44E52", edgecolor="white")
ax1.axhline(y=max_entropy, color="gray", linestyle="--", alpha=0.7, label=f"Max entropy ({max_entropy:.2f})")
ax1.set_xlabel("Embedding Dimension")
ax1.set_ylabel("Mean Output Entropy")
ax1.set_title("Output Entropy vs Width", fontweight="bold")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# Parameter count vs width
ax2.bar([str(w) for w in width_values], [p / 1000 for p in width_param_counts], color="#8172B2", edgecolor="white")
ax2.set_xlabel("Embedding Dimension")
ax2.set_ylabel("Parameters (thousands)")
ax2.set_title("Parameter Count vs Width", fontweight="bold")
ax2.grid(axis="y", alpha=0.3)

# Annotate
for i, (e, p) in enumerate(zip(width_entropies, width_param_counts)):
    ax1.text(i, e + 0.02, f"{e:.3f}", ha="center", va="bottom", fontsize=9)
    ax2.text(i, p / 1000 + 1, f"{p:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

print("The parameter count grows rapidly with width due to the quadratic scaling.")

In [ ]:
# Compare output probability distributions across depths
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, n_l in zip(axes, depth_values):
    torch.manual_seed(42)
    m = GPT2(VOCAB_SIZE, EMBED_DIM, N_HEADS, n_l, MAX_SEQ_LEN).to(device)
    m.eval()
    with torch.no_grad():
        depth_logits = m(test_ids[:1])
        depth_probs = F.softmax(depth_logits[0, -1], dim=-1).cpu().numpy()

    ax.bar(range(VOCAB_SIZE), depth_probs, width=1.0, color="#4C72B0", alpha=0.7)
    ax.set_title(f"Depth = {n_l} layers", fontweight="bold")
    ax.set_xlabel("Token ID")
    ax.set_ylabel("Probability")
    ax.set_xlim(0, VOCAB_SIZE)
    ax.axhline(y=1.0 / VOCAB_SIZE, color="red", linestyle="--", alpha=0.5, label="Uniform")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Next-Token Probability Distributions (Untrained, Last Position)",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

print("Even without training, deeper models tend to produce slightly more peaked distributions.")
print("The random weight interactions across layers create structure in the output.")

In [ ]:
# --- 9.3 Activation Norms: Depth Comparison ---
# Compare how activation norms evolve for models with different depths.

fig, ax = plt.subplots(figsize=(12, 5))

colors_depth = ["#4C72B0", "#DD8452", "#55A868"]

for color, n_l in zip(colors_depth, [1, 2, 4]):
    torch.manual_seed(42)
    m = GPT2(VOCAB_SIZE, EMBED_DIM, N_HEADS, n_l, MAX_SEQ_LEN).to(device)
    m.eval()

    norms = []

    with torch.no_grad():
        t_emb = m.token_emb(probe_input)
        p_emb = m.pos_emb(torch.arange(probe_input.shape[1], device=device))
        x_trace = t_emb + p_emb
        norms.append(x_trace.norm().item())

        for i, blk in enumerate(m.blocks):
            x_trace = blk(x_trace)
            norms.append(x_trace.norm().item())

        x_trace = m.ln_f(x_trace)
        norms.append(x_trace.norm().item())

    ax.plot(range(len(norms)), norms, marker="o", color=color, label=f"{n_l} layers", linewidth=2)

ax.set_xlabel("Layer")
ax.set_ylabel("Activation L2 Norm")
ax.set_title("Activation Norms Through Models of Different Depths", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Residual connections cause a steady growth in activation norms.")
print("Deeper models accumulate more, but the growth is controlled (approximately linear).")

---
## Part 10: Weight Tying Impact and Attention Patterns

In [ ]:
# Compare parameter counts with and without weight tying
print(f"{'Model':<16} {'With tying':>14} {'Without tying':>14} {'Savings':>14} {'Savings %':>10}")
print("-" * 72)

for cfg in configs:
    with_tying = count_params(cfg)["total"]
    extra = cfg.vocab_size * cfg.embed_dim  # the output head weights
    without_tying = with_tying + extra
    savings = extra
    pct = 100.0 * savings / without_tying
    print(f"{cfg.name:<16} {with_tying:>14,} {without_tying:>14,} {savings:>14,} {pct:>9.1f}%")

print("\nWeight tying saves significant parameters, especially for smaller models")
print("where the embedding matrix is a larger fraction of total parameters.")

In [ ]:
# Extract and visualize causal attention patterns
def get_attention_weights(mdl, inp_ids):
    """Extract attention weights from each block."""
    mdl.eval()
    all_attn_weights = []

    with torch.no_grad():
        B, T = inp_ids.shape
        t_emb = mdl.token_emb(inp_ids)
        p_emb = mdl.pos_emb(torch.arange(T, device=inp_ids.device))
        x_a = t_emb + p_emb

        for blk in mdl.blocks:
            # Manually compute attention to extract weights
            ln_x = blk.ln_1(x_a)
            B_, T_, C_ = ln_x.shape
            qkv = blk.attn.c_attn(ln_x)
            q, k, v = qkv.split(blk.attn.embed_dim, dim=2)
            n_h = blk.attn.n_heads
            h_d = blk.attn.head_dim

            q = q.view(B_, T_, n_h, h_d).transpose(1, 2)
            k = k.view(B_, T_, n_h, h_d).transpose(1, 2)

            scale = 1.0 / (h_d ** 0.5)
            attn_w = (q @ k.transpose(-2, -1)) * scale
            cmask = torch.triu(torch.ones(T_, T_, device=x_a.device), diagonal=1).bool()
            attn_w = attn_w.masked_fill(cmask.unsqueeze(0).unsqueeze(0), float("-inf"))
            attn_w = F.softmax(attn_w, dim=-1)  # (B, H, T, T)

            all_attn_weights.append(attn_w.cpu())

            # Complete the forward pass for this block
            x_a = blk(x_a)

    return all_attn_weights


short_input = torch.randint(0, VOCAB_SIZE, (1, 16), device=device)
attn_weights = get_attention_weights(model, short_input)

# Plot attention patterns for each head in each block
n_blocks = len(attn_weights)
fig, axes = plt.subplots(n_blocks, N_HEADS, figsize=(3 * N_HEADS, 3 * n_blocks))

if n_blocks == 1:
    axes = axes.reshape(1, -1)

for block_idx in range(n_blocks):
    for head_idx in range(N_HEADS):
        ax = axes[block_idx, head_idx]
        attn_map = attn_weights[block_idx][0, head_idx].numpy()  # (T, T)
        ax.imshow(attn_map, cmap="Blues", vmin=0, vmax=attn_map.max())
        ax.set_title(f"Block {block_idx}, Head {head_idx}", fontsize=9)
        if head_idx == 0:
            ax.set_ylabel("Query position")
        if block_idx == n_blocks - 1:
            ax.set_xlabel("Key position")

fig.suptitle("Causal Attention Patterns (Untrained Model)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("The upper triangle is always zero (causal mask).")
print("Different heads attend to different positions, even without training.")

---
## Part 11: Scaling Behavior Summary

Let us create a final comprehensive plot showing how parameters scale across all four GPT-2 variants.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = [c.name.replace("GPT-2 ", "") for c in configs]
n_layers_list = [c.n_layers for c in configs]
dims_list = [c.embed_dim for c in configs]
totals = [count_params(c)["total"] for c in configs]

# Plot 1: Total params
ax = axes[0]
bar_objs = ax.bar(names, [t / 1e6 for t in totals],
                  color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"], edgecolor="white")
ax.set_ylabel("Parameters (millions)")
ax.set_title("Total Parameter Count", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for bar_obj, t in zip(bar_objs, totals):
    lbl = f"{t/1e9:.2f}B" if t >= 1e9 else f"{t/1e6:.0f}M"
    ax.text(bar_obj.get_x() + bar_obj.get_width() / 2, bar_obj.get_height() + 20,
            lbl, ha="center", va="bottom", fontsize=9, fontweight="bold")

# Plot 2: Params vs embed_dim (showing quadratic relationship)
ax = axes[1]
ax.scatter(dims_list, [t / 1e6 for t in totals], s=100, color="#DD8452", zorder=5)
d_range = np.linspace(min(dims_list) * 0.8, max(dims_list) * 1.1, 100)
coeffs = np.polyfit(dims_list, [t / 1e6 for t in totals], 2)
ax.plot(d_range, np.polyval(coeffs, d_range), "--", color="gray", alpha=0.7, label="Quadratic fit")
for d, t, name in zip(dims_list, totals, names):
    lbl = f"{t/1e9:.2f}B" if t >= 1e9 else f"{t/1e6:.0f}M"
    ax.annotate(f"{name}\n{lbl}", (d, t / 1e6), textcoords="offset points",
                xytext=(10, 10), fontsize=8)
ax.set_xlabel("Embedding Dimension")
ax.set_ylabel("Parameters (millions)")
ax.set_title("Params vs Width", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: Attention vs FFN params per block
ax = axes[2]
attn_per_block = [4 * c.embed_dim**2 + 4 * c.embed_dim for c in configs]
ffn_per_block = [8 * c.embed_dim**2 + 5 * c.embed_dim for c in configs]
x_pos = np.arange(len(configs))
bw = 0.35
ax.bar(x_pos - bw / 2, [a / 1e6 for a in attn_per_block], bw, label="Attention", color="#4C72B0")
ax.bar(x_pos + bw / 2, [f / 1e6 for f in ffn_per_block], bw, label="FFN", color="#55A868")
ax.set_xticks(x_pos)
ax.set_xticklabels(names)
ax.set_ylabel("Params per block (millions)")
ax.set_title("Attention vs FFN (per block)", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

fig.suptitle("GPT-2 Scaling Behavior", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

print("Key observation: FFN always has ~2x the parameters of attention per block.")
print("This is because FFN has 8*D^2 params vs attention's 4*D^2 params (ignoring biases).")

---
## Key Insights

### Architecture

1. **GPT-2 is remarkably simple.** The entire model is: embeddings, a stack of identical transformer blocks, a final LayerNorm, and a linear output head. No cross-attention, no encoder -- just a decoder.

2. **Pre-norm residual connections** (`x = x + sublayer(LN(x))`) are critical. They keep the residual stream clean, allowing gradients to flow directly through the network. This is what makes training deep transformers stable.

3. **Weight tying** between the token embedding and output head reduces parameters (especially significant for smaller models) and creates a natural symmetry: tokens embedded close together in the input space also compete for similar output probabilities.

### Scaling

4. **Parameters scale as ~12LD^2** (approximately). The dominant terms come from the attention and FFN weight matrices, which are quadratic in the embedding dimension.

5. **FFN is the biggest component** -- roughly 2/3 of each transformer block's parameters live in the feed-forward network (8D^2 for FFN vs 4D^2 for attention).

6. **Embedding share shrinks with scale.** For GPT-2 Small, embeddings are ~30% of parameters. For GPT-2 XL, they are ~5%. This is because embeddings scale as V*D (linear in D) while blocks scale as L*D^2.

### Ablations

7. **Depth vs width trade-off.** Adding layers increases parameters linearly. Doubling embed_dim quadruples per-block parameters. In practice, depth is a more parameter-efficient way to add capacity, which is why GPT-2 XL has 48 layers rather than just a very wide 12-layer model.

8. **Activation norms grow with depth** due to residual connections accumulating contributions from each block. The final LayerNorm re-normalizes before the output projection, preventing the logits from being dominated by the accumulated scale.

9. **Even random models have structure.** The causal mask, combined with the multi-head architecture, creates distinct attention patterns per head even before any training. Training sharpens these patterns into semantically meaningful attention.